# CNN — Detecção de Células Cervicais (SipakMed) v3

**Dataset:** SipakMed — 5 classes de células do colo do útero  
**Modelo:** MobileNetV2 com Transfer Learning  
**Output:** `cnn_cervical.h5` para uso no Streamlit

### Melhorias v3
- **CLAHE** — equalização adaptativa de histograma por canal
- **EarlyStopping em `val_auc`** (v2 usava `val_recall` — causava parada prematura)
- **Data augmentation moderado** — rotation 20°, sem vertical_flip, brightness ±10%
- **Melhor checkpoint** — salva o modelo com maior val_auc, não o do último epoch

### Pré-requisito
Faça upload do dataset SipakMed no Google Drive antes de rodar.
```
MyDrive/sipakmed/
  im_Dyskeratotic/im_Dyskeratotic/*.bmp
  im_Koilocytotic/im_Koilocytotic/*.bmp
  im_Metaplastic/im_Metaplastic/*.bmp
  im_Parabasal/im_Parabasal/*.bmp
  im_Superficial-Intermediate/im_Superficial-Intermediate/*.bmp
```
### Ativar GPU: **Runtime → Change runtime type → T4 GPU → Save**

## 1. Monta o Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

SIPAKMED_PATH = '/content/drive/MyDrive/sipakmed'
OUTPUT_PATH   = '/content/drive/MyDrive/cervical_artifacts'
os.makedirs(OUTPUT_PATH, exist_ok=True)

assert os.path.exists(SIPAKMED_PATH), f'SipakMed não encontrado em: {SIPAKMED_PATH}'
classes = sorted([d for d in os.listdir(SIPAKMED_PATH) if os.path.isdir(os.path.join(SIPAKMED_PATH, d))])
print(f'Classes: {classes}')

## 2. Instala dependências e imports

In [ ]:
!pip install -q opencv-python-headless

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 3. Pré-processamento com CLAHE

In [ ]:
def apply_clahe(img_uint8):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    channels = cv2.split(img_uint8)
    eq_channels = [clahe.apply(ch) for ch in channels]
    return cv2.merge(eq_channels)


def preprocess_image(fpath, img_size=(224, 224)):
    img = cv2.imread(fpath)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, img_size)
    img = apply_clahe(img)
    img = img.astype(np.float32) / 255.0
    return img


# Visualiza efeito do CLAHE
sample_path = os.path.join(SIPAKMED_PATH, 'im_Dyskeratotic', 'im_Dyskeratotic')
sample_file = os.path.join(sample_path, sorted(os.listdir(sample_path))[0])
img_orig = cv2.resize(cv2.cvtColor(cv2.imread(sample_file), cv2.COLOR_BGR2RGB), (224, 224))
img_clahe = apply_clahe(img_orig)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img_orig);  axes[0].set_title('Original');   axes[0].axis('off')
axes[1].imshow(img_clahe); axes[1].set_title('Com CLAHE'); axes[1].axis('off')
plt.suptitle('Efeito do CLAHE', fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Carrega as imagens

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

BINARY_MAP = {
    'im_Dyskeratotic':             1,
    'im_Koilocytotic':             1,
    'im_Metaplastic':              0,
    'im_Parabasal':                0,
    'im_Superficial-Intermediate': 0,
}

images, labels = [], []

for class_folder, label in BINARY_MAP.items():
    class_path = os.path.join(SIPAKMED_PATH, class_folder, class_folder)
    if not os.path.isdir(class_path):
        print(f'  Pasta não encontrada: {class_path}')
        continue
    bmp_files = [f for f in os.listdir(class_path) if f.endswith('.bmp')]
    print(f'  {class_folder}: {len(bmp_files)} imagens')
    for fname in bmp_files:
        img = preprocess_image(os.path.join(class_path, fname), IMG_SIZE)
        if img is not None:
            images.append(img)
            labels.append(label)

X = np.array(images, dtype=np.float32)
y = np.array(labels, dtype=np.int32)
print(f'\nTotal: {len(X)} | Normais: {(y==0).sum()} | Anômalas: {(y==1).sum()}')

## 5. Split e Data Augmentation

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
print(f'Treino: {len(X_train)} | Validação: {len(X_val)} | Teste: {len(X_test)}')

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True,
    vertical_flip=False,
    brightness_range=[0.9, 1.1],
    fill_mode='nearest'
)
train_gen = datagen.flow(X_train, y_train, batch_size=BATCH_SIZE)
print(f'Steps por epoch: {len(X_train) // BATCH_SIZE}')

## 6. Constrói o modelo

In [ ]:
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.Recall(name='recall'), keras.metrics.AUC(name='auc')]
)
model.summary()

## 7. Treinamento — Fase 1

In [ ]:
n_normal = (y_train == 0).sum()
n_anomal = (y_train == 1).sum()
total    = len(y_train)
class_weight = {0: total / (2 * n_normal), 1: total / (2 * n_anomal)}
print(f'Class weights: {class_weight}')

# IMPORTANTE: monitor='val_auc' — evita parar cedo por recall instável
callbacks = [
    EarlyStopping(monitor='val_auc', patience=10, restore_best_weights=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1),
    ModelCheckpoint(
        filepath=os.path.join(OUTPUT_PATH, 'cnn_cervical_best.h5'),
        monitor='val_auc', save_best_only=True, mode='max', verbose=1
    )
]

print('=== FASE 1 (base congelada) ===')
history1 = model.fit(
    train_gen, epochs=30,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=1
)

## 8. Fine-tuning — Fase 2

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-6),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.Recall(name='recall'), keras.metrics.AUC(name='auc')]
)

print('=== FASE 2 (fine-tuning últimas 50 camadas) ===')
history2 = model.fit(
    train_gen, epochs=20,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=1
)

## 9. Avaliação

In [ ]:
print('=== AVALIAÇÃO FINAL ===')
test_loss, test_acc, test_recall, test_auc = model.evaluate(X_test, y_test, verbose=0)
print(f'Accuracy: {test_acc:.4f} | Recall: {test_recall:.4f} | AUC: {test_auc:.4f}')

y_pred_proba = model.predict(X_test).ravel()

# Testa os dois thresholds para comparar
for thresh in [0.5, 0.35]:
    y_pred = (y_pred_proba >= thresh).astype(int)
    print(f'\nThreshold={thresh}:')
    print(classification_report(y_test, y_pred, target_names=['Normal', 'Anomala']))

In [ ]:
# Matriz de confusão com threshold=0.5
y_pred = (y_pred_proba >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Anomala'], yticklabels=['Normal', 'Anomala'])
plt.title('Matriz de Confusão')
plt.xlabel('Predito'); plt.ylabel('Real')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
acc  = history1.history['accuracy']     + history2.history['accuracy']
vacc = history1.history['val_accuracy'] + history2.history['val_accuracy']
rec  = history1.history['recall']       + history2.history['recall']
vrec = history1.history['val_recall']   + history2.history['val_recall']
axes[0].plot(acc, label='Treino'); axes[0].plot(vacc, label='Validação')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(rec, label='Treino'); axes[1].plot(vrec, label='Validação')
axes[1].set_title('Recall'); axes[1].legend()
plt.suptitle('Curvas de Treinamento v2', fontweight='bold')
plt.tight_layout(); plt.show()

## 10. Testa as melhores imagens de exemplo por classe

In [ ]:
# Encontra a imagem com maior confiança de cada classe
# Rode esta célula APÓS o treino para saber quais imagens usar como exemplo no Streamlit

BINARY_MAP_LABEL = {
    'im_Dyskeratotic':             'Anomala',
    'im_Koilocytotic':             'Anomala',
    'im_Metaplastic':              'Normal',
    'im_Parabasal':                'Normal',
    'im_Superficial-Intermediate': 'Normal',
}

THRESHOLD = 0.5

print('Buscando melhores imagens de exemplo por classe...\n')
best_per_class = {}

for folder, label in BINARY_MAP_LABEL.items():
    path = os.path.join(SIPAKMED_PATH, folder, folder)
    bmps = sorted([f for f in os.listdir(path) if f.endswith('.bmp')])

    best_conf  = 0
    best_fname = None

    for fname in bmps:
        img = preprocess_image(os.path.join(path, fname))
        if img is None:
            continue
        prob = model.predict(np.expand_dims(img, 0), verbose=0)[0][0]
        pred = 'Anomala' if prob >= THRESHOLD else 'Normal'
        conf = prob * 100 if prob >= THRESHOLD else (1 - prob) * 100

        if pred == label and conf > best_conf:
            best_conf  = conf
            best_fname = fname

    best_per_class[folder] = (best_fname, best_conf)
    status = f'{best_fname} ({best_conf:.1f}%)' if best_fname else 'NENHUMA CORRETA'
    print(f'  {folder}: {status}')

print('\n=== RESUMO ===')
for folder, (fname, conf) in best_per_class.items():
    print(f'  {folder}: {fname} — {conf:.1f}%')

In [ ]:
# Visualiza as melhores imagens de cada classe
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, (folder, (fname, conf)) in zip(axes, best_per_class.items()):
    if fname:
        path = os.path.join(SIPAKMED_PATH, folder, folder, fname)
        img = cv2.resize(cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB), (224, 224))
        ax.imshow(img)
        label = BINARY_MAP_LABEL[folder]
        ax.set_title(f'{folder.replace("im_", "")}\n{label} {conf:.1f}%', fontsize=8)
    else:
        ax.set_title(f'{folder}\nNENHUMA', fontsize=8)
    ax.axis('off')

plt.suptitle('Melhores imagens de exemplo por classe', fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Salva o modelo

In [ ]:
import shutil

# Usa o melhor checkpoint (melhor val_auc durante o treino)
# em vez do modelo restaurado pelo EarlyStopping no último epoch
best_path  = os.path.join(OUTPUT_PATH, 'cnn_cervical_best.h5')
model_path = os.path.join(OUTPUT_PATH, 'cnn_cervical.h5')

shutil.copy(best_path, model_path)

size_mb = os.path.getsize(model_path) / (1024 * 1024)
print(f'Modelo salvo: {model_path} ({size_mb:.1f} MB)')
print('\n✅ Coloque em: app/models/artifacts/cnn_cervical.h5')

In [ ]:
from google.colab import files
files.download(model_path)